In [1]:
import os
import pandas as pd
import glob

ODDS_DIR  = r"C:\Users\semwi\FPL-Core-Insights\data\odds"
OUTPUT    = r"C:\Users\semwi\FPL-Core-Insights\data\odds\all_odds_combined.csv"

# Alle odds kolommen
ODDS_COLS = [
    'B365H','B365D','B365A',
    'BWH','BWD','BWA',
    'IWH','IWD','IWA',
    'LBH','LBD','LBA',
    'PSH','PSD','PSA',
    'WHH','WHD','WHA',
    'SJH','SJD','SJA',
    'VCH','VCD','VCA',
    'BbMxH','BbAvH','BbMxD','BbAvD','BbMxA','BbAvA',
    'BbMx>2.5','BbAv>2.5','BbMx<2.5','BbAv<2.5',
    'PSCH','PSCD','PSCA',
]

all_dfs = []
for path in sorted(glob.glob(os.path.join(ODDS_DIR, "E0*.csv"))):
    df = pd.read_csv(path, encoding='latin-1')
    
    # Basiskolommen altijd meenemen
    keep = ['Date', 'HomeTeam', 'AwayTeam']
    # Alleen odds kolommen die in dit bestand bestaan
    keep += [c for c in ODDS_COLS if c in df.columns]
    
    df = df[keep].copy()
    df['source_file'] = os.path.basename(path)
    all_dfs.append(df)
    print(f"  {os.path.basename(path)}: {len(df)} rijen, {len(df.columns)} kolommen")

combined = pd.concat(all_dfs, ignore_index=True)
combined['Date'] = pd.to_datetime(combined['Date'], dayfirst=True, errors='coerce')
combined = combined.sort_values('Date').reset_index(drop=True)

combined.to_csv(OUTPUT, index=False)
print(f"\nKlaar! {len(combined)} rijen → {OUTPUT}")

  E0 (1).csv: 309 rijen, 19 kolommen
  E0 (10).csv: 380 rijen, 38 kolommen
  E0 (11).csv: 380 rijen, 38 kolommen
  E0 (12).csv: 381 rijen, 41 kolommen
  E0 (2).csv: 380 rijen, 19 kolommen
  E0 (3).csv: 380 rijen, 25 kolommen
  E0 (4).csv: 380 rijen, 25 kolommen
  E0 (5).csv: 380 rijen, 25 kolommen
  E0 (6).csv: 380 rijen, 25 kolommen
  E0 (7).csv: 380 rijen, 25 kolommen
  E0 (8).csv: 380 rijen, 35 kolommen
  E0 (9).csv: 380 rijen, 38 kolommen

Klaar! 4490 rijen → C:\Users\semwi\FPL-Core-Insights\data\odds\all_odds_combined.csv


In [2]:
import os
import pandas as pd
import glob

ODDS_DIR = r"C:\Users\semwi\FPL-Core-Insights\data\odds raw"
POLY_PATH = r"C:\Users\semwi\FPL-Core-Insights\data\polymarket_snapshots.csv"

TEAM_MAP = {
    "Man United":              "Manchester United",
    "Man City":                "Manchester City",
    "Tottenham":               "Tottenham Hotspur",
    "Newcastle":               "Newcastle United",
    "Brighton":                "Brighton & Hove Albion",
    "West Ham":                "West Ham United",
    "Leicester":               "Leicester City",
    "Ipswich":                 "Ipswich Town",
    "Luton":                   "Luton Town",
    "Wolves":                  "Wolverhampton",
    "Nott'm Forest":           "Nottingham Forest",
    "Leeds":                   "Leeds United",
    "West Brom":               "West Bromwich Albion",
}

# Laad odds 24/25 en 25/26
all_dfs = []
for path in sorted(glob.glob(os.path.join(ODDS_DIR, "E0*.csv"))):
    df = pd.read_csv(path, encoding='latin-1')
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    # Alleen seizoenen vanaf aug 2024
    df = df[df['Date'] >= '2024-08-01']
    if len(df) > 0:
        all_dfs.append(df)
        print(f"  {os.path.basename(path)}: {len(df)} rijen")

odds = pd.concat(all_dfs, ignore_index=True)
odds['HomeTeam'] = odds['HomeTeam'].replace(TEAM_MAP)
odds['AwayTeam'] = odds['AwayTeam'].replace(TEAM_MAP)
odds['_date'] = odds['Date'].dt.date.astype(str)

# Laad polymarket en zet om naar odds
poly = pd.read_csv(POLY_PATH)
poly['kickoff'] = pd.to_datetime(poly['kickoff'])
poly = poly[poly['kickoff'] >= '2024-08-01'].copy()
poly['_date'] = poly['kickoff'].dt.date.astype(str)
poly['odds_home_poly'] = (1 / poly['prob_home']).round(2)
poly['odds_draw_poly'] = (1 / poly['prob_draw']).round(2)
poly['odds_away_poly'] = (1 / poly['prob_away']).round(2)

# Merge
merged = poly.merge(
    odds[['_date','HomeTeam','AwayTeam','B365H','B365D','B365A','PSH','PSD','PSA']],
    left_on=['_date','home_team','away_team'],
    right_on=['_date','HomeTeam','AwayTeam'],
    how='inner'
).drop(columns=['HomeTeam','AwayTeam','_date'])

print(f"\nGekoppeld: {len(merged)} wedstrijden")
print()

# Vergelijking
merged['diff_home'] = (merged['odds_home_poly'] - merged['B365H']).round(2)
merged['diff_draw'] = (merged['odds_draw_poly'] - merged['B365D']).round(2)
merged['diff_away'] = (merged['odds_away_poly'] - merged['B365A']).round(2)

print("=== GEMIDDELD VERSCHIL (Polymarket odds - Bet365 odds) ===")
print(f"Thuis:     {merged['diff_home'].mean():+.3f}")
print(f"Gelijkspel:{merged['diff_draw'].mean():+.3f}")
print(f"Uit:       {merged['diff_away'].mean():+.3f}")
print("(positief = Polymarket geeft hogere odds dan Bet365)")

print()
print("=== GEMIDDELDE ODDS VERGELIJKING ===")
print(f"{'':20s} {'Polymarket':>12} {'Bet365':>10} {'Pinnacle':>10}")
print(f"{'Thuis':20s} {merged['odds_home_poly'].mean():>12.3f} {merged['B365H'].mean():>10.3f} {merged['PSH'].mean():>10.3f}")
print(f"{'Gelijkspel':20s} {merged['odds_draw_poly'].mean():>12.3f} {merged['B365D'].mean():>10.3f} {merged['PSD'].mean():>10.3f}")
print(f"{'Uit':20s} {merged['odds_away_poly'].mean():>12.3f} {merged['B365A'].mean():>10.3f} {merged['PSA'].mean():>10.3f}")

print()
print("=== MARGE (som impliciete kansen) ===")
merged['marge_poly']  = (1/merged['odds_home_poly'] + 1/merged['odds_draw_poly'] + 1/merged['odds_away_poly'])
merged['marge_b365']  = (1/merged['B365H'] + 1/merged['B365D'] + 1/merged['B365A'])
merged['marge_pinn']  = (1/merged['PSH']   + 1/merged['PSD']   + 1/merged['PSA'])
print(f"Polymarket: {merged['marge_poly'].mean():.4f}  ({(merged['marge_poly'].mean()-1)*100:.2f}% marge)")
print(f"Bet365:     {merged['marge_b365'].mean():.4f}  ({(merged['marge_b365'].mean()-1)*100:.2f}% marge)")
print(f"Pinnacle:   {merged['marge_pinn'].mean():.4f}  ({(merged['marge_pinn'].mean()-1)*100:.2f}% marge)")

  E0 (1).csv: 309 rijen
  E0 (2).csv: 380 rijen

Gekoppeld: 676 wedstrijden

=== GEMIDDELD VERSCHIL (Polymarket odds - Bet365 odds) ===
Thuis:     +0.258
Gelijkspel:+0.258
Uit:       +0.563
(positief = Polymarket geeft hogere odds dan Bet365)

=== GEMIDDELDE ODDS VERGELIJKING ===
                       Polymarket     Bet365   Pinnacle
Thuis                       2.936      2.677      2.740
Gelijkspel                  4.431      4.173      4.330
Uit                         4.695      4.132      4.351

=== MARGE (som impliciete kansen) ===
Polymarket: 1.0025  (0.25% marge)
Bet365:     1.0553  (5.53% marge)
Pinnacle:   1.0358  (3.58% marge)


C:\Users\semwi\AppData\Local\Temp\ipykernel_22196\2434061332.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
C:\Users\semwi\AppData\Local\Temp\ipykernel_22196\2434061332.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
C:\Users\semwi\AppData\Local\Temp\ipykernel_22196\2434061332.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  o